# Hands-on RL Fine-tuning with verl on AMD Instinct™ MI300X

## Qwen3-4B LoRA GRPO Training on a Single GPU

In this workshop, we will run a small but complete RL fine-tuning workflow for a reasoning-style language model.

Modern LLMs are moving from chatbots to reasoning agents.  To support reasoning, tool use, and multi-step decision making, we use reinforcement learning after pretraining and supervised fine-tuning.

We will train **Qwen3-4B** with **LoRA + GRPO** using **verl** and **vLLM** on a single **AMD Instinct™ MI300X GPU**.

| Component | Role |
|---|---|
| **Qwen3-4B** | Base model |
| **LoRA** | Lightweight fine-tuning method |
| **GRPO** | Reward-based RL algorithm |
| **verl** | RL training framework |
| **vLLM** | Fast rollout generation engine |
| **AMD MI300X** | Single-GPU training platform |

## Why RL Workloads Fit AMD Instinct GPUs

Reinforcement learning workloads differ from ordinary supervised fine-tuning.

In RL post-training, the model repeatedly generates responses, scores those responses, and updates the policy. This means rollout generation can become a major part of the workload.

AMD Instinct MI300X is useful for this workshop because it provides:

- large HBM memory headroom
- high memory bandwidth
- strong compute capability
- ROCm software support
- enough room for model weights, LoRA adapters, rollout generation, and training overhead

This makes MI300X a strong fit for a single-GPU hands-on RL training demo.


## How verl and vLLM Work Together

In this training job, **verl controls the RL loop**, while **vLLM handles rollout generation**.

 | Component | Responsibility |
  |---|---|
  | **verl** | Orchestrates the RL loop: rollout scheduling, advantage computation, PPO/GRPO loss, optimizer step, weight sync, logging, checkpointing |
  | **vLLM** | High-throughput rollout engine — generates a *group* of responses per prompt for GRPO |
  | **Actor (LoRA)** | The policy being trained. Only LoRA adapter weights are updated; the base model stays frozen |
  | **Reference model** | Frozen copy of the initial policy, used as the KL anchor to keep the actor from drifting |
  | **GRPO** | Computes a group-relative advantage per response (per-prompt baseline), then applies a PPO-style clipped objective |
  | **Reward function** | Scores each generated response (rule-based, model-based, or a mix) |

### Runtime Flow

```text
 Prompt batch
    ↓
  vLLM rollout engine generates G responses per prompt (group sampling)
    ↓
  Reward function scores each response
    ↓
  GRPO computes group-relative advantages (per-prompt baseline)
    ↓
  Training engine recomputes log-probs; PPO-clipped loss + KL-to-reference
    ↓
  Optimizer updates LoRA adapters on the actor
    ↓
  verl syncs updated LoRA weights into the vLLM engine
    ↓
  Next training step

## Workshop Runtime Stack

| Layer | Setting |
|---|---|
| Model | `Qwen/Qwen3-4B` |
| Dataset | GSM8K |
| RL algorithm | `algorithm.adv_estimator=grpo` |
| Fine-tuning | LoRA — `actor_rollout_ref.model.lora_rank=32`, `actor_rollout_ref.model.lora_alpha=64` |
| Rollout backend | `actor_rollout_ref.rollout.name=vllm` |
| Number of samples | `actor_rollout_ref.rollout.n=8` |
| Tensor parallel size | `actor_rollout_ref.rollout.tensor_model_parallel_size=1` |
| Framework version | verl v0.7.1 |
| Rollout runtime | vLLM 0.18.0 with ROCm 7.2.1 |
  | Hardware | AMD Instinct MI300X (gfx942), single GPU |


 ## Step 1: Check AMD GPU

  Before launching verl, confirm the AMD GPUs are visible inside the container.

  Run `amd-smi list` to enumerate the devices. You should see one or more entries with:
  - **GPU-Name** containing `MI300X` (architecture `gfx942`)
  - ROCm version `7.2.1` reported in the header
  - **Mem-Usage** mostly free (e.g. `~3 GB / 196592 MB` per GPU when idle)

In [4]:
!amd-smi

+------------------------------------------------------------------------------+
| AMD-SMI 26.0.0+37d158ab      amdgpu version: 6.14.14  ROCm version: 7.0.0    |
| Platform: Linux Baremetal                                                    |
|-------------------------------------+----------------------------------------|
| BDF                        GPU-Name | Mem-Uti   Temp   UEC       Power-Usage |
| GPU  HIP-ID  OAM-ID  Partition-Mode | GFX-Uti    Fan               Mem-Usage |
|=====================================+========================================|
| 0000:0c:00.0 ...Instinct MI300X OAM | 0 %      49 °C   0           169/750 W |
|   0       0       2        SPX/NPS1 | 0 %        N/A          4373/196592 MB |
|-------------------------------------+----------------------------------------|
| 0000:22:00.0 ...Instinct MI300X OAM | 0 %      48 °C   0           170/750 W |
|   1       1       1        SPX/NPS1 | 0 %        N/A          3015/196592 MB |
|---------------------------

## Step 2: Verify verl Installation

  This workshop runs inside the prebuilt `rocm/verl` container, which already ships verl as an editable install at `/workspace/verl`.

  The cell below confirms the install and prints the version. The workshop targets **verl 0.7.1**.

  The training entrypoint used later is:

  ```bash
  python3 -m verl.trainer.main_ppo
  ```

  (GRPO and PPO share this entrypoint; the algorithm is selected via `algorithm.adv_estimator=grpo`.)

In [8]:
import importlib, sys

try:
  import verl
  print("verl version :", verl.__version__)
  print("verl module  :", verl.__file__)
except ImportError:
  sys.exit(
      "verl is not installed. This notebook expects the rocm/verl image "
  )

verl version : 0.7.0.dev
verl module  : /workspace/verl/verl/__init__.py


 ## Step 3: Check vLLM and ROCm Runtime

  verl uses vLLM as the rollout engine on top of PyTorch + ROCm. The cell below confirms that:

  - **vLLM** is importable
  - **PyTorch** is the ROCm build (HIP runtime present)
  - At least one **AMD GPU** is visible to PyTorch

  If any of these fail, the training launch in Step 8 will not work.

  The relevant rollout config used later is:

  ```bash
  actor_rollout_ref.rollout.name=vllm
  actor_rollout_ref.rollout.tensor_model_parallel_size=1
  actor_rollout_ref.rollout.n=8
  ```

In [9]:
import sys

import vllm
import torch

hip_version = getattr(torch.version, "hip", None)
device_count = torch.cuda.device_count() if torch.cuda.is_available() else 0

print("vLLM version    :", vllm.__version__)
print("PyTorch version :", torch.__version__)
print("HIP runtime     :", hip_version or "NOT a ROCm build")
print("GPUs visible    :", device_count)

assert hip_version, "PyTorch is not built with ROCm/HIP — install the ROCm wheel."
assert device_count > 0, "No AMD GPUs visible — check --device=/dev/dri --device=/dev/kfd."

# torch.cuda.get_device_name() returns '' for MI300X on some ROCm builds; use amd-smi for the marketing name.
print("OK: vLLM + ROCm runtime ready.")

vLLM version    : 0.11.0.dev
PyTorch version : 2.9.0a0+git1c57644
HIP runtime     : 7.0.51831-a3e329ad8
GPUs visible    : 8
OK: vLLM + ROCm runtime ready.


  ## Step 4: Prepare the GSM8K Dataset

  We use **GSM8K** (grade-school math word problems) because it gives us a *deterministic* reward: each example ends with a `#### <number>` marker, so the GRPO reward function just extracts the model's final number and
  string-compares it to the gold answer. No reward model needed — perfect for a workshop.

  verl ships a preprocessing script at `examples/data_preprocess/gsm8k.py` that downloads GSM8K from Hugging Face and writes `train.parquet` / `test.parquet` for the trainer to consume.

  The output paths below (`/workspace/data/gsm8k/`) are what Step 7 will reference via `data.train_files=` and `data.val_files=`.

In [11]:
%%bash
set -e

VERL_DIR=/workspace/verl
DATA_DIR=/workspace/data/gsm8k

if [ -f "${DATA_DIR}/train.parquet" ] && [ -f "${DATA_DIR}/test.parquet" ]; then
echo "GSM8K already prepared at ${DATA_DIR}, skipping."
else
mkdir -p "${DATA_DIR}"
python3 "${VERL_DIR}/examples/data_preprocess/gsm8k.py" --local_dir "${DATA_DIR}"
fi

echo "Files in ${DATA_DIR}:"
ls -lh "${DATA_DIR}"

GSM8K already prepared at /workspace/data/gsm8k, skipping.
Files in /workspace/data/gsm8k:
total 3.9M
-rw-r--r-- 1 root root 605K Apr  9 02:34 test.parquet
-rw-r--r-- 1 root root 3.3M Apr  9 02:34 train.parquet


## Step 5: Model and Dataset Paths

We point the trainer at two things:

- **Dataset** — the GSM8K parquets produced by Step 4, on the container's local filesystem.
- **Model** — Qwen3-4B base weights, pulled from Hugging Face into a host-mounted cache (`/root/.cache/huggingface` → `/data/huggingface` on the host) so they survive container restarts.


In [14]:
from pathlib import Path

MODEL_PATH  = "Qwen/Qwen3-4B"
VERL_DIR    = Path("/workspace/verl")
DATA_DIR    = Path("/workspace/data/gsm8k")
train_files = str(DATA_DIR / "train.parquet")
test_files  = str(DATA_DIR / "test.parquet")

print("MODEL_PATH  :", MODEL_PATH)
print("VERL_DIR    :", VERL_DIR, "(exists:", VERL_DIR.exists(), ")")
print("train_files :", train_files, "(exists:", Path(train_files).exists(), ")")
print("test_files  :", test_files,  "(exists:", Path(test_files).exists(),  ")")

assert VERL_DIR.exists(), f"verl not found at {VERL_DIR} — re-check Step 2."

MODEL_PATH  : Qwen/Qwen3-4B
VERL_DIR    : /workspace/verl (exists: True )
train_files : /workspace/data/gsm8k/train.parquet (exists: True )
test_files  : /workspace/data/gsm8k/test.parquet (exists: True )


##### dataset: download if missing, then peek:

In [17]:
import subprocess, sys
import pandas as pd

if not (Path(train_files).exists() and Path(test_files).exists()):
  print("GSM8K parquets not found — running the verl preprocessing script.")
  DATA_DIR.mkdir(parents=True, exist_ok=True)
  # Pulls openai/gsm8k from Hugging Face Datasets and writes train/test parquets.
  subprocess.run(
      [sys.executable, str(VERL_DIR / "examples/data_preprocess/gsm8k.py"),
       "--local_dir", str(DATA_DIR)],
      check=True,
  )
else:
  print(f"GSM8K parquets already present at {DATA_DIR}, skipping download.")

train_df = pd.read_parquet(train_files)
test_df  = pd.read_parquet(test_files)
print(f"\ntrain rows : {len(train_df):>6}   columns: {list(train_df.columns)}")
print(f"test  rows : {len(test_df):>6}   columns: {list(test_df.columns)}")

# Show one example so the reader sees what the trainer will consume.
example = train_df.iloc[0]
print("\n--- sample row ---")
for col in train_df.columns:
  val = str(example[col])
  print(f"{col}: {val[:300]}{'...' if len(val) > 300 else ''}")

GSM8K parquets already present at /workspace/data/gsm8k, skipping download.

train rows :   7473   columns: ['data_source', 'prompt', 'ability', 'reward_model', 'extra_info']
test  rows :   1319   columns: ['data_source', 'prompt', 'ability', 'reward_model', 'extra_info']

--- sample row ---
data_source: openai/gsm8k
prompt: [{'content': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May? Let\'s think step by step and output the final answer after "####".', 'role': 'user'}]
ability: math
reward_model: {'ground_truth': '72', 'style': 'rule'}
extra_info: {'answer': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72', 'index': 0, 'question': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altog...


##### model: download via from_pretrained if not in HF cache:

In [19]:
import os
from pathlib import Path

# HF stores snapshots under hub/models--<org>--<name>/.
hf_cache  = Path(os.environ.get("HF_HOME", str(Path.home() / ".cache/huggingface"))) / "hub"
model_dir = hf_cache / f"models--{MODEL_PATH.replace('/', '--')}"

if not model_dir.exists() or not any(model_dir.rglob("*.safetensors")):
  print(f"{MODEL_PATH} not found in HF cache — downloading via transformers.")
  print("First run pulls ~8 GB; subsequent runs hit the cache.\n")

  import transformers
  # from_pretrained downloads config, tokenizer, and weight shards into the HF cache,
  # then materializes the model object. We keep it on CPU and discard immediately —
  # the goal here is only to populate the cache for the trainer in Step 8.
  tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_PATH)
  model     = transformers.AutoModelForCausalLM.from_pretrained(
      MODEL_PATH,
      torch_dtype="auto",
      device_map="cpu",
  )
  del model, tokenizer
else:
  print(f"{MODEL_PATH} already cached at {model_dir}, skipping download.")

# Show what's in the snapshot so the reader sees what an LLM repo actually contains.
snapshot_root = next((model_dir / "snapshots").iterdir(), None)
if snapshot_root is not None:
  print(f"\nFiles in snapshot ({snapshot_root.name}):")
  for p in sorted(snapshot_root.iterdir()):
      size_mb = p.stat().st_size / (1024 * 1024) if p.is_file() else 0
      print(f"  {p.name:<40} {size_mb:>8.1f} MB" if p.is_file() else f"  {p.name}/")

Qwen/Qwen3-4B already cached at /root/.cache/huggingface/hub/models--Qwen--Qwen3-4B, skipping download.

Files in snapshot (1cfa9a7208912126459214e8b04321603b3df60c):
  .gitattributes                                0.0 MB
  LICENSE                                       0.0 MB
  README.md                                     0.0 MB
  config.json                                   0.0 MB
  generation_config.json                        0.0 MB
  merges.txt                                    1.6 MB
  model-00001-of-00003.safetensors           3774.5 MB
  model-00002-of-00003.safetensors           3802.7 MB
  model-00003-of-00003.safetensors             95.0 MB
  model.safetensors.index.json                  0.0 MB
  tokenizer.json                               10.9 MB
  tokenizer_config.json                         0.0 MB
  vocab.json                                    2.6 MB


## Step 6: Understand the Key Training Configuration

Step 7 launches `verl.trainer.main_ppo` with a long list of Hydra overrides. Here are the ones that actually shape this run; everything else is plumbing.

### Algorithm

| Config | Meaning |
|---|---|
| `algorithm.adv_estimator=grpo` | Compute group-relative advantages instead of GAE — this is what makes the run GRPO instead of vanilla PPO |
| `actor_rollout_ref.actor.use_kl_loss=true` | Add a KL penalty against the frozen reference model (anchors the policy, prevents collapse) |
| `actor_rollout_ref.actor.kl_loss_coef=0.001` | Strength of that KL penalty |

### Model and LoRA

| Config | Meaning |
|---|---|
| `actor_rollout_ref.model.path=Qwen/Qwen3-4B` | Base model — pulled from the HF cache populated in Step 5 |
| `actor_rollout_ref.model.lora_rank=32` | Rank of the LoRA matrices A (d × 32) and B (32 × d). Higher = more capacity, more trainable params |
| `actor_rollout_ref.model.lora_alpha=64` | LoRA scaling. Effective scale = `alpha / rank = 2.0` |

### Rollout (vLLM)

| Config | Meaning |
|---|---|
| `actor_rollout_ref.rollout.name=vllm` | Use vLLM as the rollout engine |
| `actor_rollout_ref.rollout.n=8` | Sample 8 responses per prompt — this is the GRPO group size |
| `actor_rollout_ref.rollout.tensor_model_parallel_size=1` | vLLM does not shard the model across GPUs |
| `actor_rollout_ref.rollout.gpu_memory_utilization=0.5` | Fraction of GPU memory vLLM may use for its KV cache. The remainder is reserved for the trainer's actor + optimizer state + activations — essential when vLLM and
training share one GPU |

### Training engine

| Config | Meaning |
|---|---|
| `actor_rollout_ref.actor.strategy=fsdp2` | Run the actor under Fully Sharded Data Parallel v2. On a single GPU there's nothing to shard across, but FSDP2 still manages activation and gradient memory efficiently |
| `actor_rollout_ref.ref.strategy=fsdp2` | Same for the frozen reference policy used by the KL term |

### Data and schedule

| Config | Meaning |
|---|---|
| `data.train_files=/workspace/data/gsm8k/train.parquet` | Training prompts — produced in Step 4 |
| `data.val_files=/workspace/data/gsm8k/test.parquet` | Eval prompts |
| `data.train_batch_size=...` | Number of prompts sampled per RL step (each is expanded into `n=8` rollouts) |
| `actor_rollout_ref.actor.optim.lr=...` | Learning rate for the LoRA adapter optimizer |
| `trainer.total_epochs=...` (or `trainer.total_training_steps=...`) | How long training runs |

## Step 7: Set Environment and Launch Qwen3-4B LoRA GRPO Training

  This step sets the minimal training environment and launches the GRPO run. Everything is in one cell so the env vars sit next to the command that consumes them.

  ### Environment

  | Variable | Purpose |
  |---|---|
  | `HIP_VISIBLE_DEVICES=0` | Restrict the trainer to GPU index 0 (the container sees 8 MI300X GPUs; this workshop uses one) |

  (verl reads the GPU count from `trainer.n_gpus_per_node=1` in the CLI, so no separate `GPUS_PER_NODE` env var is needed.)

  ### What spins up at launch

  When the cell runs, verl will:

  1. Initialize Ray (single-node) and spawn worker processes
  2. Load **Qwen3-4B** as the actor under FSDP2 and attach LoRA adapters (rank 32)
  3. Load a frozen **reference model** copy under FSDP2 for the KL anchor
  4. Boot a **vLLM** rollout engine inside the actor worker
  5. Run a baseline eval pass (`val_before_train=True`)
  6. Enter the GRPO training loop: rollout → reward → advantages → PPO-clipped + KL update → sync LoRA weights into vLLM → next step
  7. Stream metrics to the console and to TensorBoard

  ### To make the run shorter (workshop time control)

  Reduce any of: `trainer.total_epochs`, `trainer.test_freq`, `data.train_batch_size`, `actor_rollout_ref.rollout.n`, `data.max_response_length`.

  > Note: `data.shuffle=False` makes the prompt order deterministic for reproducibility in this workshop. For real training runs, set it to `True`.

In [ ]:
%%bash
set -e

# Pin the trainer to a single MI300X. 
export HIP_VISIBLE_DEVICES=0

cd /workspace/verl

MODEL_PATH="Qwen/Qwen3-4B"
train_files="/workspace/data/gsm8k/train.parquet"
test_files="/workspace/data/gsm8k/test.parquet"

GPU_MEMORY_UTILIZATION=0.5
max_token_len_per_gpu=24576

# Single-GPU workshop configuration
TP_VALUE=1
TRAIN_BATCH_SIZE=16
MINI_BATCH_SIZE=16

python3 -m verl.trainer.main_ppo \
  algorithm.adv_estimator=grpo \
  trainer.val_before_train=True \
  data.train_files="${train_files}" \
  data.val_files="${test_files}" \
  data.train_batch_size=${TRAIN_BATCH_SIZE} \
  data.max_prompt_length=1024 \
  data.max_response_length=1024 \
  data.filter_overlong_prompts=True \
  data.truncation='error' \
  data.shuffle=False \
  actor_rollout_ref.model.path="${MODEL_PATH}" \
  actor_rollout_ref.model.use_remove_padding=True \
  actor_rollout_ref.model.enable_gradient_checkpointing=True \
  actor_rollout_ref.model.lora_rank=32 \
  actor_rollout_ref.model.lora_alpha=64 \
  actor_rollout_ref.actor.optim.lr=1.0e-05 \
  actor_rollout_ref.actor.use_dynamic_bsz=True \
  actor_rollout_ref.actor.ppo_mini_batch_size=${MINI_BATCH_SIZE} \
  actor_rollout_ref.actor.ppo_max_token_len_per_gpu=${max_token_len_per_gpu} \
  actor_rollout_ref.actor.use_kl_loss=True \
  actor_rollout_ref.actor.kl_loss_coef=0.001 \
  actor_rollout_ref.actor.kl_loss_type=low_var_kl \
  actor_rollout_ref.actor.entropy_coeff=0 \
  actor_rollout_ref.actor.strategy=fsdp2 \
  actor_rollout_ref.actor.fsdp_config.model_dtype=bf16 \
  actor_rollout_ref.actor.fsdp_config.param_offload=False \
  actor_rollout_ref.actor.fsdp_config.optimizer_offload=False \
  actor_rollout_ref.rollout.log_prob_use_dynamic_bsz=True \
  actor_rollout_ref.rollout.log_prob_max_token_len_per_gpu=${max_token_len_per_gpu} \
  actor_rollout_ref.rollout.tensor_model_parallel_size=${TP_VALUE} \
  actor_rollout_ref.rollout.name=vllm \
  actor_rollout_ref.rollout.gpu_memory_utilization=${GPU_MEMORY_UTILIZATION} \
  actor_rollout_ref.rollout.n=8 \
  actor_rollout_ref.rollout.load_format=safetensors \
  actor_rollout_ref.rollout.layered_summon=True \
  actor_rollout_ref.ref.log_prob_use_dynamic_bsz=True \
  actor_rollout_ref.ref.log_prob_max_token_len_per_gpu=${max_token_len_per_gpu} \
  actor_rollout_ref.ref.fsdp_config.param_offload=True \
  actor_rollout_ref.ref.strategy=fsdp2 \
  actor_rollout_ref.ref.fsdp_config.model_dtype=bf16 \
  algorithm.use_kl_in_reward=False \
  trainer.use_legacy_worker_impl=disable \
  trainer.critic_warmup=0 \
  trainer.logger='["console","tensorboard"]' \
  trainer.project_name='grpo_qwen_llm' \
  trainer.experiment_name='qwen3_4b_grpo_lora_rocm_single_gpu' \
  trainer.n_gpus_per_node=1 \
  trainer.nnodes=1 \
  trainer.save_freq=-1 \
  trainer.test_freq=10 \
  trainer.total_epochs=1

## Step 9: Monitor Training Logs

The training command logs to the console and TensorBoard.

Typical signals to watch:

- rollout generation speed
- reward value
- response length
- KL value
- policy loss
- entropy
- validation score

If the workshop is time-limited, we only need to confirm that the pipeline runs and produces a few training iterations.


In [ ]:
%%bash
set -e

cd "${HOME}/verl-workshop/verl"

echo "Possible TensorBoard log locations:"
find . -maxdepth 4 -type d \( -name "tensorboard" -o -name "runs" -o -name "logs" \) 2>/dev/null || true

echo ""
echo "To start TensorBoard manually, you can run:"
echo "tensorboard --logdir . --host 0.0.0.0 --port 6006"